# Surf Spot Search Taiwan: Application of IR and System Design

This notebook documents the **information retrieval (IR) approach** and the **system design** of *Surf Spot Search Taiwan*, based on the implementation in `notebooks/00`–`07`. It covers only these two aspects. User interface, evaluation procedures and setup are documented elsewhere (`README.md`).

> **Code cells** contain excerpts copied from the project notebooks (the source notebook is named in each cell). They are self-contained and run on their own, so the notebook can be executed with *Run All*.
> **Diagrams** use Mermaid. They render in JupyterLab 4 and on GitHub; in VS Code a Mermaid-enabled Markdown extension may be needed.

## 1. Introduction

*Surf Spot Search Taiwan* is a domain-specific search system that answers English natural-language questions such as *"beginner-friendly beach break near Kaohsiung this weekend"* with a ranked list of Taiwanese surf spots. Each result states whether and when the spot is surfable for the user's skill level.

The system combines two kinds of evidence. **Textual relevance** comes from a corpus of 35 sourced spot profiles. **Situational relevance** comes from hourly wave and wind forecasts and buoy observations. Two rankers are implemented on the same corpus and interface: a **baseline** (lexical BM25) and an **improved, condition-aware** ranker. The improved ranker adds query understanding, vocabulary expansion, a forecast-based condition-fit score and template explanations.

## 2. Application of Information Retrieval

### Retrieval Problem

The task is **ad-hoc ranking of a small, closed document collection** (35 spot profiles) for short, informal English queries. It differs from classic document retrieval in three ways:

1. **Relevance is time-dependent.** The same query has different correct answers on different days, because a spot is only useful if its forecast conditions fit the requested time window.
2. **Relevance depends on the user.** A 1.5 m swell is ideal for an intermediate surfer but unsafe for a beginner.
3. **There is a vocabulary gap.** Users write surf slang (*beachie*, *newbie*) and spell places in different romanisations (*Kaohsiung / Gaoxiong / Takao*), while the profiles use standard terms.

Formally, the system estimates $rel(q, d, t)$ for query $q$, spot $d$ and time $t$. It combines a topical component (text) with a situational component (conditions).

### Data Collection

| Source | Content | Role in retrieval | Implementation |
|---|---|---|---|
| **Spot profiles** (`data/spots/spots.csv`) | 35 spots: location, region, break/bottom type, orientation, swell window, ideal tide, skill levels, hazards, facilities, description, sources | **Document collection** (text) and spot attributes for condition scoring | `01_spot_corpus` (validation) |
| **Open-Meteo Marine API** | Hourly 7-day forecast: wave, swell and wind-wave height/period/direction; sea level incl. tides | Situational evidence | `00_data_collection` |
| **Open-Meteo Weather API** | Hourly wind speed, direction, gusts; temperature | Situational evidence (offshore/onshore wind) | `00_data_collection` |
| **CWA Open Data** | `O-B0075-001`: 48-h buoy and tide-station observations; `O-B0076-001`: station list with English names | Forecast correction and forecast-error evaluation | `00_data_collection` |
| **Lexical resources** (`data/resources/`) | Surf-slang synonyms (38), gazetteer of places with spelling variants (38), region phrases, rule thresholds | Query processing and condition scoring | `02`, `03`, `04` |

The dynamic sources are downloaded once per day and stored as a **dated snapshot** (`data/snapshots/YYYY-MM-DD/`, Parquet plus a manifest). Retrieval always runs against a snapshot, so results are **reproducible**: relevance judgments and experiments refer to fixed data instead of a moving live forecast.

In [ ]:
# Excerpt from notebooks/00_data_collection.ipynb: the external data sources and the variables requested
MARINE_URL = "https://marine-api.open-meteo.com/v1/marine"
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
MARINE_VARS = [
    "wave_height", "wave_direction", "wave_period",
    "swell_wave_height", "swell_wave_direction", "swell_wave_period",
    "wind_wave_height", "wind_wave_direction", "wind_wave_period",
    "sea_level_height_msl",
]
WIND_VARS = ["wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "temperature_2m"]

CWA_URL = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/{dataset_id}"
CWA_DATASETS = {"obs_48h": "O-B0075-001"}        # 48-hour buoy and tide-station observations
CWA_STATIONS_URL = "https://opendata.cwa.gov.tw/fileapi/v1/opendataapi/O-B0076-001"   # station list, English names

**Document representation.** Each profile is turned into one text document. Structured fields are rendered as words (e.g. crowd level *low* → *"uncrowded quiet"*), so attribute queries can match them lexically. The beach orientation is deliberately **not** rendered as text: *"faces east"* would make a north-coast spot match the query *east coast*. Orientation is used only by the condition score.

In [ ]:
# Excerpt from notebooks/02_indexing.ipynb: from spot profile to indexable document
CROWD_WORDS = {"low": "uncrowded quiet", "medium": "", "high": "crowded busy", "unknown": ""}


def build_doc_text(spot):
    parts = [
        spot["name"], " ".join(spot.get("alt_names") or []),
        f"{spot['region']} coast", f"near {spot.get('nearest_town') or ''}",
        f"{spot['break_type']} break", f"{spot['bottom_type']} bottom",
        "for " + " ".join(spot.get("skill_levels") or []),
        CROWD_WORDS.get(spot.get("crowd_level") or "unknown", ""),
        "hazards: " + ", ".join(spot.get("hazards") or []),
        "facilities: " + ", ".join(spot.get("facilities") or []),
        spot.get("description") or "",
    ]
    return ". ".join(p for p in parts if p and p.strip())

### Query Processing

Query processing has two layers.

**(1) Text analysis**, identical for documents and queries: lower-casing, replacing hyphens and apostrophes with spaces, tokenisation, stop-word removal and Snowball stemming. The scikit-learn English stop-word list is used **minus domain words** such as *east*, *south* and *bottom*, which carry meaning in surfing (*east coast*, *sand bottom*).

In [ ]:
# Excerpt from notebooks/02_indexing.ipynb: the shared text analyser
import re
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

DOMAIN_KEEP = {"east", "west", "north", "south", "bottom", "back", "front", "top", "full", "down", "up",
               "off", "over", "empty", "fire", "less", "few", "first", "side", "well"}
STOPWORDS = frozenset(ENGLISH_STOP_WORDS) - DOMAIN_KEEP
_stemmer = SnowballStemmer("english")
_TOKEN_RE = re.compile(r"[a-z0-9]+")


def normalize_text(text):
    return re.sub(r"[-'’_/]", " ", str(text).lower())


def tokenize(text):
    return _TOKEN_RE.findall(normalize_text(text))


def analyze(text):
    return [_stemmer.stem(t) for t in tokenize(text) if t not in STOPWORDS]


analyze("Uncrowded reef-break on the East coast, sandy bottom; beginners welcome!")

**(2) Query understanding** (improved system only) extracts structured constraints with transparent rules; no trained model is used:

- **Location.** Region phrases (`regions.csv`, longest phrase first) map to regions. Word n-grams (3→1) are fuzzy-matched (rapidfuzz ratio ≥ 88) against gazetteer places with spelling variants and against spot names, which handles *Gaoxiong* and typos such as *Kaohsuing*.
- **Skill level.** Regular-expression lexicons, e.g. *first time*, *newbie* → beginner.
- **Time window.** Rules for *now, today, tomorrow, this/next weekend, weekday names, next N days, morning/afternoon, dawn patrol*, plus explicit dates via `dateparser`. They are resolved relative to the **snapshot reference time**, not the system clock, which keeps evaluation reproducible.
- **Condition intent.** Whether the query asks about conditions at all; this decides the fusion weight (see *Retrieval & Ranking*).

Missing constraints fall back to explicit defaults: all of Taiwan, the next 48 hours, intermediate level.

In [ ]:
# Excerpt from notebooks/04_search.ipynb: query understanding entry point
# (parse_location, parse_time, SKILL_PATTERNS and CONDITION_PATTERN are defined in the same notebook)
def parse_query(query, ref_time):
    text = normalize_query(query)
    pq = ParsedQuery(raw=query, text=text)
    parse_location(text, pq)
    for skill, pattern in SKILL_PATTERNS.items():
        if re.search(pattern, text):
            pq.skill, pq.skill_from_query = skill, True
            break
    pq.windows, pq.time_label, pq.time_from_query = parse_time(text, pd.Timestamp(ref_time))
    if not pq.windows:
        pq.notes.append("The requested time has already passed in this snapshot.")
    pq.condition_intent = bool(pq.time_from_query or pq.skill_from_query or re.search(CONDITION_PATTERN, text))
    return pq

**Vocabulary expansion.** Phrases from the curated synonym list (sourced from Wiktionary and the Wikipedia *Glossary of surfing*) add the stems of their expansions with a **weight of 0.5**. Expansion therefore improves recall without outweighing the user's own terms.

In [ ]:
# Excerpt from notebooks/04_search.ipynb: weighted query terms with slang expansion
def query_stems(pq, expand):
    weights = {s: 1.0 for s in analyze(pq.text)}
    added = []
    if expand:
        for phrase, exps in SYNONYMS.items():
            if re.search(rf"\b{re.escape(phrase)}\b", pq.text):
                for e in exps:
                    for s in e["stems"]:
                        if weights.get(s, 0) < e["weight"]:
                            weights[s] = e["weight"]
                    added.append(e["expansion"])
    return weights, added

### Retrieval & Ranking

**Lexical retrieval (both systems).** Documents are scored with **Okapi BM25** ($k_1 = 1.2$, $b = 0.75$). The IDF of `rank_bm25`'s Okapi variant becomes negative for terms that occur in more than half of the documents, which is common in a small, homogeneous corpus (*beach*, *break*). The project therefore uses **Lucene's non-negative IDF**:

$$\mathrm{IDF}(t) = \log\left(1 + \frac{N - n_t + 0.5}{n_t + 0.5}\right)$$

The **baseline** ranks by this BM25 score alone, on the raw query. In the improved system, BM25 scores of the weighted (expanded) query terms are summed and divided by the corpus maximum, giving a text score in $[0, 1]$. A spot named explicitly in the query receives text score 1.

In [ ]:
# Excerpt from notebooks/02_indexing.ipynb and 04_search.ipynb: BM25 with Lucene IDF and weighted query scoring
import math
from rank_bm25 import BM25Okapi


class BM25Lucene(BM25Okapi):
    # BM25 with Lucene's non-negative IDF

    def _calc_idf(self, nd):
        self.idf = {word: math.log(1 + (self.corpus_size - freq + 0.5) / (freq + 0.5)) for word, freq in nd.items()}


def bm25_scores(weights):
    bm25 = BM25_INDEX["bm25"]
    scores = np.zeros(len(DOC_IDS))
    for stem, w in weights.items():
        scores += w * bm25.get_scores([stem])
    return scores

**Filtering.** Extracted locations act as **hard filters** before ranking:
- spots within **40 km** (haversine distance) of a recognised place, or within the recognised regions, are kept;
- spots named in the query are always kept;
- if nothing remains, the filter is relaxed and the user is told so.

```python
# Excerpt from notebooks/04_search.ipynb (search_verbose)
if pq.place_latlon:
    distance = _haversine_km(pq.place_latlon[0], pq.place_latlon[1], SPOT_LAT, SPOT_LON)
    keep = distance <= NEAR_RADIUS_KM
elif pq.regions:
    keep = np.isin(SPOT_REGION, pq.regions)
if pq.named_spots:
    keep |= np.isin(DOC_IDS, pq.named_spots)
```

**Condition fit (CF).** For every daylight hour in the requested window, six factor scores in $[0, 1]$ are computed from the snapshot and the spot profile:
- **Size:** wave height against skill-specific limits; 1 inside the ideal range, stepped down in 0.1 steps outside it, 0 when flat or unsafe.
- **Period:** a floor value for short wind swell, rising to 1 for organised swell.
- **Direction:** swell inside the spot's window scores 1; outside it, the score steps down to a floor of 0.1.
- **Wind:** the offshore/onshore component relative to the beach orientation.
- **Tide:** the tide stage derived from sea level, compared with the spot's preferred stage.
- **Level:** whether the spot profile lists the user's skill level.

The factors are combined **multiplicatively**, so a single disqualifying factor dominates:

$$CF = \text{size}^{w_1}\cdot\text{period}^{w_2}\cdot\text{direction}^{w_3}\cdot\text{wind}^{w_4}\cdot\text{tide}^{0.5}\cdot\text{level}$$

The spot's CF is the CF of its **best hour** in the window. All thresholds and weights are read from `surfability_thresholds.csv`, each with a cited source or a documented design choice. If a nearby buoy observation (≤ 3 h old) differs from the forecast by more than 30 %, the forecast wave height for the next 12 hours is scaled towards the observation.

In [ ]:
# Excerpt from notebooks/03_surfability.ipynb: the stepped size score, run with the project's intermediate thresholds
import numpy as np
import pandas as pd

SCORE_STEP = 0.1


def stepped(x):
    return np.floor(np.asarray(x, dtype=float) / SCORE_STEP + 1e-9) * SCORE_STEP


def stepped_size(h, low_limit, ideal_low, ideal_high, high_limit):
    h = np.asarray(h, dtype=float)
    ideal_low, ideal_high = max(ideal_low, low_limit), min(ideal_high, high_limit)
    with np.errstate(divide="ignore", invalid="ignore"):
        below = stepped(0.1 + 0.9 * (h - low_limit) / (ideal_low - low_limit))
        above = stepped(0.1 + 0.9 * (high_limit - h) / (high_limit - ideal_high))
    return np.select([h < low_limit, h < ideal_low, h <= ideal_high, h <= high_limit],
                     [0.0, below, 1.0, above], default=0.0)


t = pd.read_csv("data/resources/surfability_thresholds.csv").set_index("parameter")["intermediate"].astype(float)
heights = [0.5, 0.6, 0.8, 1.0, 1.5, 1.7, 1.9]
pd.DataFrame({"wave height (m)": heights,
              "size score (intermediate)": stepped_size(heights, t["wave_min_m"], t["wave_ideal_low_m"],
                                                        t["wave_ideal_high_m"], t["wave_max_m"]).round(1)})

**Score fusion and final ranking.** The improved system combines both components with a **weighted linear fusion**:

$$\text{score}(q, d, t) = \alpha \cdot \text{text}(q, d) + (1 - \alpha) \cdot CF(d, t, \text{skill})$$

- $\alpha = 0.5$ when the query expresses condition intent (time, skill or condition words);
- $\alpha = 0.85$ for pure attribute queries such as *"reef break with showers"*;
- $\alpha$ can be tuned on development queries (`TUNE_ALPHA` in notebook 05).

Spots that are unsafe for the user's level (waves above the maximum, or a profile that excludes the level) receive a **−1 penalty**, which moves them to the bottom with a warning. The top-$k$ results are returned together with a **template explanation** generated only from the computed values of the best hour, so the explanation cannot claim anything the scoring did not compute.

```python
# Excerpt from notebooks/04_search.ipynb (search_verbose)
a = alpha if alpha is not None else (ALPHA if pq.condition_intent else ALPHA_NO_CONDITION_INTENT)
row["score"] = a * row["text_score"] + (1 - a) * row["cf"]
if SAFETY_DEMOTE and unsafe:
    row["score"] -= 1.0
```

**System variants.** All rankers share one `search()` function and differ only in configuration flags. This enables ablation experiments: switching off condition fit, query understanding or synonym expansion, plus an *oracle* that scores with observed instead of forecast conditions. An optional hybrid variant (BM25 + `bge-small-en-v1.5` dense retrieval fused with reciprocal rank fusion) is implemented but disabled by default.

In [ ]:
# Excerpt from notebooks/04_search.ipynb: baseline, improved system and ablation variants
SYSTEMS = {
    "baseline":        {"qu": False, "syn": False, "cf": False, "dense": False, "oracle": False},
    "improved":        {"qu": True,  "syn": True,  "cf": True,  "dense": False, "oracle": False},
    "improved_no_cf":  {"qu": True,  "syn": True,  "cf": False, "dense": False, "oracle": False},
    "improved_no_qu":  {"qu": False, "syn": True,  "cf": True,  "dense": False, "oracle": False},
    "improved_no_syn": {"qu": True,  "syn": False, "cf": True,  "dense": False, "oracle": False},
    "improved_hybrid": {"qu": True,  "syn": True,  "cf": True,  "dense": True,  "oracle": False},
    "improved_oracle": {"qu": True,  "syn": True,  "cf": True,  "dense": False, "oracle": True},
}
pd.DataFrame(SYSTEMS).T

### IR Workflow Diagram

```mermaid
flowchart LR
    Q["English query"] --> A["Text analysis<br/>tokenise · stop words · stem"]
    Q --> U["Query understanding<br/>place · region · skill · time · intent"]
    A --> X["Slang expansion<br/>weight 0.5"]
    X --> B["BM25 · Lucene IDF<br/>text score 0–1"]
    U --> F["Location filter<br/>40 km · region · named spot"]
    B --> F
    F --> C["Condition fit per spot<br/>best hour in time window"]
    S[("Daily snapshot<br/>forecast + buoys")] --> C
    T[("Thresholds CSV")] --> C
    C --> R["Fusion<br/>α·text + (1−α)·CF<br/>− safety penalty"]
    U -. "α 0.5 / 0.85" .-> R
    R --> K["Top-k ranking<br/>+ template explanation"]
    B -. "baseline: BM25 only" .-> K2["Baseline ranking"]
```

## 3. System Design

### Architecture Overview

The system is a **notebook-based batch-and-query pipeline**. Each stage is a numbered Jupyter notebook that owns one responsibility and exchanges data with the others **through files on disk** (Parquet snapshots, a validated JSON corpus, pickled indexes). Online components load earlier notebooks as libraries with `%run`. In this *library mode* (environment variable `SURF_IR_LIBRARY_MODE=1`), a loaded notebook defines its functions but skips its own build steps, so no code is duplicated. A small script (`collect_snapshot.py`) runs the collection notebook on a daily schedule.

```mermaid
flowchart TB
    subgraph EXT["External services"]
        OM1["Open-Meteo Marine API"]
        OM2["Open-Meteo Weather API"]
        CWA["CWA Open Data<br/>O-B0075-001 · O-B0076-001"]
    end
    subgraph OFF["Offline / scheduled"]
        SCH["collect_snapshot.py<br/>(daily scheduler)"] --> N00["00 data collection"]
        N01["01 spot corpus<br/>validation"]
        N02["02 indexing<br/>BM25 · synonyms"]
    end
    subgraph STORE["File storage"]
        SNAP[("data/snapshots/<br/>YYYY-MM-DD")]
        CORP[("data/spots · data/resources")]
        PROC[("data/processed")]
        IDX[("indexes/")]
    end
    subgraph ON["Query time"]
        N03["03 surfability<br/>condition fit"]
        N04["04 search<br/>query understanding · ranking"]
        N05["05 evaluation"]
        N06["06 demo"]
    end
    OM1 --> N00
    OM2 --> N00
    CWA --> N00
    N00 --> SNAP
    CORP --> N01 --> PROC --> N02 --> IDX
    SNAP --> N03
    PROC --> N03
    IDX --> N04
    N03 --> N04
    N04 --> N05
    N04 --> N06
```

### Main Components

| Component | Responsibility | Main artefacts |
|---|---|---|
| **00 Data collection** | Fetches marine and wind forecasts per spot and CWA observations; maps stations to English names; writes an atomic daily snapshot with a manifest; retries and logs failures | `data/snapshots/YYYY-MM-DD/*.parquet`, `manifest.json` |
| **01 Spot corpus** | Loads and validates the 35 profiles (coordinates, angles, controlled vocabularies, sources) | `data/processed/spots_clean.json` |
| **02 Indexing** | Shared text analyser, document construction, BM25 index, compiled synonym list, optional dense index | `indexes/bm25.pkl`, `indexes/synonyms.json` |
| **03 Surfability** | Snapshot loading and caching, per-hour factor scores, condition fit, buoy correction, explanations | `condition_fit()`, `factor_scores()` |
| **04 Search** | Query understanding, expansion, retrieval, filtering, fusion, system variants; **single `search()` interface** | `search()`, `search_verbose()` |
| **05 / 06** | Consumers of `search()` (evaluation, demo) | — |
| **Lexical and rule resources** | Gazetteer, regions, synonyms, thresholds as editable, sourced CSV files | `data/resources/*.csv` |

### Data Flow

Data moves along two paths. The **offline path** prepares the collection and the daily evidence. The **query path** joins them at query time.

```mermaid
flowchart LR
    subgraph Offline
        direction TB
        P["spots.csv"] -->|validate| J["spots_clean.json"]
        J -->|build_doc_text · analyze| I["BM25 index"]
        R["surf_synonyms.csv"] -->|stem| SY["synonyms.json"]
        API["Open-Meteo + CWA"] -->|daily fetch| SN["snapshot<br/>marine · wind · obs"]
    end
    subgraph Query["Query time"]
        direction TB
        Q["query + snapshot date"] --> PQ["parsed query<br/>place · skill · window"]
        PQ --> TS["text scores"]
        PQ --> CF["condition fit<br/>per spot"]
        TS --> FU["fused score · ranking"]
        CF --> FU
        FU --> OUT["ranked spots<br/>+ explanations"]
    end
    I --> TS
    SY --> TS
    J --> CF
    SN --> CF
    G["gazetteer · regions"] --> PQ
    TH["thresholds"] --> CF
```

A snapshot is loaded once and cached in memory, and so are the per-spot hourly frames; repeated queries therefore require no disk access. For *live* use, the same collection function builds an in-memory snapshot instead of a saved one.

### Design Decisions

| Decision | Rationale |
|---|---|
| **Daily snapshots instead of live calls at query time** | Forecasts change hourly. Fixed snapshots make relevance judgments and system comparisons reproducible, and make the system independent of API availability. |
| **BM25 baseline with Lucene IDF** | A standard, transparent lexical baseline. Lucene's IDF avoids negative weights for frequent terms in a small collection. |
| **Rule-based query understanding (no ML model)** | About 50 test queries are too few to train or validate a model. Rules are inspectable, deterministic and resolved against the snapshot time. |
| **Separate topical and situational evidence, fused linearly** | Keeps both components interpretable and allows ablation. A single weight $\alpha$ can be tuned on development queries; learning-to-rank was rejected given the data size. |
| **Multiplicative condition fit with stepped factor scores** | One disqualifying factor (flat, unsafe, blocked swell) cannot be compensated by others. The 0.1 steps make the score breakdown readable. |
| **All thresholds in a sourced CSV** | Domain parameters are separated from code; each value carries a source or a documented design choice, and can be revised without code changes. |
| **Hard location filter with relaxation** | Geographic constraints are treated as requirements, but an empty result is never returned. |
| **Template explanations** | Explanations are built from computed values only, guaranteeing faithfulness. |
| **One `search()` interface with configuration flags** | The evaluation and the demo run exactly the same code; baseline, improved system and ablations differ only in flags. |
| **Notebooks as modules (`%run` in library mode) and file-based hand-off** | Each pipeline stage can be developed, run and owned independently, without copying code between notebooks. |
| **Offshore forecast points and same-coast buoys** | The coarse marine grid can place a beach in a land cell; profiles can specify an offshore point, and buoys are matched only within 60 km. |

## 4. Conclusion

*Surf Spot Search Taiwan* applies classical IR, namely BM25 retrieval over a curated document collection with stemming, stop-word handling and query expansion, to a problem in which **relevance depends on time, place and user skill**. The improved ranker extends the lexical model with rule-based query understanding, geographic filtering and a forecast-driven condition-fit score. It fuses topical and situational evidence linearly and explains each result from the computed factors.

The system is designed as a modular, file-based notebook pipeline. Daily snapshots of external forecasts and observations make retrieval reproducible, and a single search interface with configurable variants supports a controlled comparison between the baseline and the improved system, including ablations of each component.